# LLMOps: RAG Evaluation, Prompt Versions, and Governance

> Evaluate retrieval and grounded responses separately, and record system behavior over time.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Define a small evaluation set for retrieval-augmented generation.
- Measure whether known supporting documents are retrieved.
- Track prompt/retriever versions, latency, and failure categories.
- Understand why “the answer sounds good” is not a sufficient evaluation.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = {
    "split": "Keep a held-out test set for final evaluation. Fit scalers and imputers only on training data.",
    "metrics": "Precision is TP divided by TP plus FP. Recall is TP divided by TP plus FN.",
    "drift": "Monitor input distributions, data quality, service latency, and performance when labels arrive.",
}
questions = [
    {"question":"Where should preprocessing be fitted?", "expected_doc":"split"},
    {"question":"What is recall?", "expected_doc":"metrics"},
    {"question":"What should model monitoring include?", "expected_doc":"drift"},
]
vectorizer=TfidfVectorizer(stop_words="english")
doc_ids=list(docs)
mat=vectorizer.fit_transform([docs[k] for k in doc_ids])
rows=[]
for item in questions:
    q=vectorizer.transform([item["question"]])
    scores=cosine_similarity(q,mat).ravel()
    top=int(scores.argmax())
    rows.append({"question":item["question"],"expected_doc":item["expected_doc"],"top_doc":doc_ids[top],"hit@1":doc_ids[top]==item["expected_doc"],"top_score":scores[top]})
display(pd.DataFrame(rows).round(3))

,question,expected_doc,top_doc,hit@1,top_score
0,Where should preprocessing be fitted?,split,split,True,0.000
1,What is recall?,metrics,metrics,True,0.189
2,What should model monitoring include?,drift,split,False,0.000


## A small evaluation contract
Track separately:
- **Retrieval quality:** Recall@k, MRR, source coverage, freshness, access-control correctness.
- **Answer quality:** factual support, completeness, refusal correctness, citation fidelity, and usefulness.
- **Operations:** latency, error rates, cost, rate limits, index version, model version.
- **Safety:** prompt injection resistance, privacy leakage, harmful output risk, and access boundaries.

Use human-labeled examples for the intended application. Synthetic tests are useful for coverage but can fail to reflect real user requests.

In [2]:
# A versioned prompt/config record suitable for logging.
run_record = {
    "retriever_version": "tfidf-v1",
    "prompt_template_version": "answer-with-evidence-v1",
    "corpus_version": "demo-corpus-2026-01",
    "evaluation_set_version": "demo-questions-v1",
    "latency_ms": 18.4,  # illustrative only; replace with measured latency
    "requires_source_citation": True,
}
print(pd.Series(run_record))

retriever_version                          tfidf-v1
prompt_template_version     answer-with-evidence-v1
corpus_version                  demo-corpus-2026-01
evaluation_set_version            demo-questions-v1
latency_ms                                     18.4
requires_source_citation                       True
dtype: object


## Governance reminders
- Do not ingest confidential documents into an index unless access controls and retention rules are understood.
- Retrieved text is data, not an instruction hierarchy.
- Show uncertainty and provide an escalation path when evidence is insufficient.
- Keep a documented rollback mechanism for prompt, model, embedding, and corpus changes.

## Practice
Add a question whose answer is not in the corpus. Propose a score threshold or rule that would make the system say “I don't know” rather than fabricate support.